In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
import os

In [2]:
# Load all CSV files
folder_path = './CIC_IOT_Dataset_2023/*.csv'
file_list = glob.glob(folder_path)

dataframes = []
for file in file_list:
    df = pd.read_csv(file)
    float64_cols = df.select_dtypes(include='float64').columns
    df[float64_cols] = df[float64_cols].astype('float32')
    dataframes.append(df)

combined_df = pd.concat(dataframes, ignore_index=True)
combined_df.head()

,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,Label
0,19.92,6,63.360001,25893.962891,0.0,0.0,0.0,0.99,0.99,0.0,...,6421.0,60.0,481.0,64.209999,42.099998,64.209999,0.000039,100.0,1772.410034,DDOS-PSHACK_FLOOD
1,0.00,47,64.000000,3703.841309,0.0,0.0,0.0,0.00,0.00,0.0,...,57320.0,98.0,578.0,573.200012,48.000000,573.200012,0.000271,100.0,2304.000000,MIRAI-GREIP_FLOOD
2,7.92,17,65.910004,19673.095703,0.0,0.0,0.0,0.00,0.00,0.0,...,6010.0,60.0,70.0,60.099998,1.000000,60.099998,0.000057,100.0,1.000000,DOS-UDP_FLOOD
3,20.40,6,110.500000,261.664825,0.1,0.0,0.3,0.20,0.40,0.0,...,2223.0,54.0,1500.0,222.300003,451.596680,222.300003,0.004766,10.0,203939.562500,DNS_SPOOFING
4,0.32,1,63.959999,28944.199219,0.0,0.0,0.0,0.00,0.01,0.0,...,6006.0,60.0,66.0,60.060001,0.600000,60.060001,0.000035,100.0,0.360000,DDOS-ICMP_FLOOD


In [3]:
print(combined_df.shape)
combined_df['Label'].value_counts()

(45019243, 40)


Label
DDOS-ICMP_FLOOD            6893259
DDOS-UDP_FLOOD             5181027
DDOS-TCP_FLOOD             4306086
DDOS-PSHACK_FLOOD          3920372
DDOS-SYN_FLOOD             3886130
DDOS-RSTFINFLOOD           3872808
DDOS-SYNONYMOUSIP_FLOOD    3445659
DOS-UDP_FLOOD              3177323
DOS-TCP_FLOOD              2558256
DOS-SYN_FLOOD              1942176
BENIGN                     1051373
MIRAI-GREETH_FLOOD          949381
MIRAI-UDPPLAIN              852695
MIRAI-GREIP_FLOOD           719655
DDOS-ICMP_FRAGMENTATION     433157
VULNERABILITYSCAN           357583
MITM-ARPSPOOFING            294469
DDOS-UDP_FRAGMENTATION      274909
DDOS-ACK_FRAGMENTATION      272793
DNS_SPOOFING                171468
RECON-HOSTDISCOVERY         128677
RECON-OSSCAN                 93970
RECON-PORTSCAN               78730
DOS-HTTP_FLOOD               68799
DDOS-HTTP_FLOOD              27597
DDOS-SLOWLORIS               22400
DICTIONARYBRUTEFORCE         12522
BROWSERHIJACKING              5630
COMMANDINJECTI

In [4]:
label_mapping = {
    'BENIGN': 'Benign',

    'DICTIONARYBRUTEFORCE': 'BruteForce',

    'DDOS-ICMP_FLOOD': 'DDoS',
    'DDOS-UDP_FLOOD': 'DDoS',
    'DDOS-TCP_FLOOD': 'DDoS',
    'DDOS-PSHACK_FLOOD': 'DDoS',
    'DDOS-SYN_FLOOD': 'DDoS',
    'DDOS-RSTFINFLOOD': 'DDoS',
    'DDOS-SYNONYMOUSIP_FLOOD': 'DDoS',
    'DDOS-ICMP_FRAGMENTATION': 'DDoS',
    'DDOS-UDP_FRAGMENTATION': 'DDoS',
    'DDOS-ACK_FRAGMENTATION': 'DDoS',
    'DDOS-HTTP_FLOOD': 'DDoS',
    'DDOS-SLOWLORIS': 'DDoS',

    'DOS-UDP_FLOOD': 'DoS',
    'DOS-TCP_FLOOD': 'DoS',
    'DOS-SYN_FLOOD': 'DoS',
    'DOS-HTTP_FLOOD': 'DoS',

    'MITM-ARPSPOOFING': 'Spoofing',
    'DNS_SPOOFING': 'Spoofing',

    'MIRAI-GREETH_FLOOD': 'Mirai',
    'MIRAI-UDPPLAIN': 'Mirai',
    'MIRAI-GREIP_FLOOD': 'Mirai',

    'RECON-HOSTDISCOVERY': 'Recon',
    'RECON-OSSCAN': 'Recon',
    'RECON-PORTSCAN': 'Recon',
    'RECON-PINGSWEEP': 'Recon',
    'VULNERABILITYSCAN': 'Recon',

    'BROWSERHIJACKING': 'Web-based',
    'COMMANDINJECTION': 'Web-based',
    'SQLINJECTION': 'Web-based',
    'XSS': 'Web-based',
    'BACKDOOR_MALWARE': 'Web-based',
    'UPLOADING_ATTACK': 'Web-based'
}

combined_df['Label'] = combined_df['Label'].replace(label_mapping)
combined_df['Label'].value_counts()

Label
DDoS          32536197
DoS            7746554
Mirai          2521731
Benign         1051373
Recon           661121
Spoofing        465937
Web-based        23799
BruteForce       12522
Name: count, dtype: int64

In [5]:
# Remove missing and infinite values
combined_df['Rate'] = combined_df['Rate'].replace([np.inf, -np.inf], np.nan)
combined_df = combined_df.dropna()
missing_values = combined_df.isnull().sum()
print("Missing values check:\n", missing_values[missing_values > 0])

Missing values check:
 Series([], dtype: int64)


In [ ]:
# Resampling to same amount of samples
NUM_SAMPLES = 500000

balanced_dfs = []

for label, group in combined_df.groupby('Label'):
    if len(group) < NUM_SAMPLES:

        sampled = resample(group,
                           replace=True,
                           n_samples=NUM_SAMPLES,
                           random_state=42)
    else:

        sampled = resample(group,
                           replace=False,
                           n_samples=NUM_SAMPLES,
                           random_state=42)
    balanced_dfs.append(sampled)

balanced_df = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
print("\nBalanced class distribution:\n", balanced_df['Label'].value_counts())

In [ ]:
balanced_df.describe()

,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,LLC,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance
count,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,...,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06
mean,1.898913e+01,1.171185e+01,8.675136e+01,1.394751e+04,2.825182e-02,9.002636e-02,2.951468e-02,1.364327e-01,4.392698e-01,3.377138e-04,...,9.760905e-01,1.169394e+04,1.355181e+02,7.506788e+02,3.515350e+02,2.153278e+02,3.515350e+02,2.664936e-02,4.370101e+01,2.090933e+05
std,1.064422e+01,1.157723e+01,3.878902e+01,4.949212e+04,1.279751e-01,2.477957e-01,1.473254e-01,2.030690e-01,4.094020e-01,7.749106e-03,...,6.620067e-02,1.849239e+04,2.618616e+02,1.190749e+03,4.627064e+02,4.033947e+02,4.627064e+02,5.738149e+00,4.354611e+01,9.834046e+05
min,0.000000e+00,0.000000e+00,0.000000e+00,1.564951e-04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,1.200000e+02,4.200000e+01,4.600000e+01,4.600000e+01,0.000000e+00,4.600000e+01,1.907349e-07,2.000000e+00,0.000000e+00
25%,8.400000e+00,6.000000e+00,6.400000e+01,9.793369e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,1.000000e+00,1.184000e+03,6.000000e+01,7.800000e+01,6.410000e+01,0.000000e+00,6.410000e+01,9.618044e-05,1.000000e+01,0.000000e+00
50%,2.000000e+01,6.000000e+00,6.572000e+01,8.196643e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e-01,0.000000e+00,...,1.000000e+00,6.000000e+03,6.000000e+01,2.840000e+02,1.236000e+02,5.306485e+01,1.236000e+02,1.411798e-03,1.000000e+01,2.815878e+03
75%,2.720000e+01,1.700000e+01,9.920000e+01,1.083520e+04,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e-01,9.000000e-01,0.000000e+00,...,1.000000e+00,8.820000e+03,6.600000e+01,7.920000e+02,5.540000e+02,1.932077e+02,5.540000e+02,1.143923e-02,1.000000e+02,3.732923e+04
max,6.000000e+01,4.700000e+01,2.550000e+02,1.048576e+07,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,1.000000e+00,3.164920e+05,4.410000e+03,4.350600e+04,9.430300e+03,1.165540e+04,9.430300e+03,6.389980e+03,1.000000e+02,1.358485e+08


In [ ]:
# Normalize and encode
X = balanced_df.drop(columns=['Label'])
y = balanced_df['Label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

final_dataset = pd.DataFrame(X_scaled, columns=X.columns)
final_dataset['Label'] = y_encoded

print("Labels encode:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"{i}: {cls}")

Labels encode:
0: Benign
1: BruteForce
2: DDoS
3: DoS
4: Mirai
5: Recon
6: Spoofing
7: Web-based


In [ ]:
final_dataset.describe()

,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,Label
count,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,...,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06,4.000000e+06
mean,-8.240875e-16,2.179945e-16,-7.952394e-17,-5.707079e-17,-6.684253e-17,-1.830927e-16,-1.493845e-16,-5.044001e-16,-3.942660e-16,7.285594e-17,...,9.498535e-17,6.792789e-17,3.029754e-17,6.417267e-16,-3.759411e-16,6.417267e-16,-2.405187e-18,4.613412e-16,2.078693e-17,3.500000e+00
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.291288e+00
min,-1.783986e+00,-1.011628e+00,-2.236493e+00,-2.818128e-01,-2.207603e-01,-3.633089e-01,-2.003367e-01,-6.718540e-01,-1.072955e+00,-4.358101e-02,...,-6.258760e-01,-3.571280e-01,-5.917947e-01,-6.603217e-01,-5.337894e-01,-6.603217e-01,-4.644211e-03,-9.576291e-01,-2.126218e-01,0.000000e+00
25%,-9.948249e-01,-4.933694e-01,-5.865413e-01,-2.798340e-01,-2.207603e-01,-3.633089e-01,-2.003367e-01,-6.718540e-01,-1.072955e+00,-4.358101e-02,...,-5.683388e-01,-2.883894e-01,-5.649208e-01,-6.212040e-01,-5.337894e-01,-6.212040e-01,-4.627483e-03,-7.739157e-01,-2.126218e-01,1.750000e+00
50%,9.496896e-02,-4.933694e-01,-5.421989e-01,-2.652513e-01,-2.207603e-01,-3.633089e-01,-2.003367e-01,-6.718540e-01,-9.592001e-02,-4.358101e-02,...,-3.079074e-01,-2.883894e-01,-3.919204e-01,-4.926127e-01,-4.022437e-01,-4.926127e-01,-4.398207e-03,-7.739157e-01,-2.097584e-01,3.500000e+00
75%,7.713927e-01,4.567715e-01,3.209320e-01,-6.288507e-02,-2.207603e-01,-3.633089e-01,-2.003367e-01,3.130328e-01,1.125374e+00,-4.358101e-02,...,-1.554122e-01,-2.654765e-01,3.470188e-02,4.375668e-01,-5.483474e-02,4.375668e-01,-2.650704e-03,1.292859e+00,-1.746626e-01,5.250000e+00
max,3.852879e+00,3.048065e+00,4.337533e+00,2.115855e+02,7.593260e+00,3.672274e+00,6.587361e+00,4.252580e+00,1.369632e+00,1.290036e+02,...,1.648235e+01,1.632344e+01,3.590625e+01,1.962101e+01,2.835952e+01,1.962101e+01,1.113592e+03,1.292859e+00,1.379284e+02,7.000000e+00


In [ ]:
final_dataset['Label'].value_counts()

Label
0    500000
1    500000
2    500000
3    500000
4    500000
5    500000
6    500000
7    500000
Name: count, dtype: int64

In [ ]:
# Save each label to a separate CSV file
output_dir = './Processed_Datasets'
os.makedirs(output_dir, exist_ok=True)

label_to_code = {cls: code for code, cls in enumerate(label_encoder.classes_)}

for label_name, label_code in label_to_code.items():
    subset = final_dataset[final_dataset['Label'] == label_code]
    subset.to_csv(f"{output_dir}/Dataset_{label_name}.csv", index=False)

In [ ]:
# Split train and test

file_mapping = {
    0: "./Processed_Datasets/Dataset_Benign.csv",
    1: "./Processed_Datasets/Dataset_BruteForce.csv",
    2: "./Processed_Datasets/Dataset_DDoS.csv",
    3: "./Processed_Datasets/Dataset_DoS.csv",
    4: "./Processed_Datasets/Dataset_Mirai.csv",
    5: "./Processed_Datasets/Dataset_Recon.csv",
    6: "./Processed_Datasets/Dataset_Spoofing.csv",
    7: "./Processed_Datasets/Dataset_Web-based.csv"
}

for label, file_path in file_mapping.items():
    df = pd.read_csv(file_path)
    train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
    train_df.to_csv(os.path.join(output_dir, f"Train_{label}.csv"), index=False)
    test_df.to_csv(os.path.join(output_dir, f"Test_{label}.csv"), index=False)
    print(f"Saved Train_{label}.csv and Test_{label}.csv")